# مقدمه‌ای بر API کروما (ChromaDB)

<div dir="rtl">
    ساخت کلاینت، ساخت Collection، پیکربندی تابع embedding، افزودن داده و انواع جستجو (فیلتر، معنایی، کلیدواژه‌ای، ترکیبی و بازرتبه‌بندی).
</div>

# فهرست مطالب
- [ ۱ - مقدمه](#1)
  - [ ۱.۱ بارگذاری کتابخانه‌های مورد نیاز](#1-1)
  - [ ۱.۲ - کلاینت ChromaDB](#1-2)
- [ ۲ - پیکربندی دیتابیس](#2)
  - [ ۲.۱ ساخت یک Collection](#2-1)
  - [ ۲.۲ پیکربندی تابع Embedding](#2-2)
  - [ ۲.۳ متادیتا (Metadata)](#2-3)
  - [ ۲.۴ افزودن عناصر به یک Collection](#2-4)
- [ ۳ - جستجو روی یک Collection](#3)
  - [ ۳.۱ فیلترها](#3-1)
  - [ ۳.۲ جستجوی معنایی](#3-2)
  - [ ۳.۳ جستجوی کلیدواژه‌ای](#3-3)
  - [ ۳.۴ جستجوی ترکیبی](#3-4)
  - [ ۳.۵ بازرتبه‌بندی](#3-5)


<a id='1'></a>
## ۱ - مقدمه
---
<a id='1-1'></a>
### ۱.۱ بارگذاری کتابخانه‌های مورد نیاز
سلول زیر را اجرا کنید تا کتابخانه‌های لازم برای این آزمایشگاه بارگذاری شوند.

In [75]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from typing import List, Dict
from tqdm import tqdm
import joblib
import chromadb
import hashlib
from pprint import pprint
import os

In [77]:
from utils import print_object_properties
from langchain_ollama import OllamaEmbeddings
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()  # متغیرهای محیطی (مثل OPENAI_API_KEY) را از فایل .env بارگذاری می‌کند


True

<a id='1-2'></a>
### ۱.۲ - کلاینت ChromaDB

<div dir="rtl">
برای کار با API کروما در این محیط، باید یک `client` بسازید. 

برای ذخیره‌سازی پایدار روی دیسک از `PersistentClient` استفاده می‌کنیم. با این کار، یک پوشه در مسیری که مشخص می‌کنید ساخته می‌شود و داده‌های شما حتی بعد از بستن نوت‌بوک نیز باقی می‌مانند.
</div>


In [79]:
client = chromadb.PersistentClient(path="./.mydb")

<div dir="rtl">
    با `client` تعریف‌شده، کارهای اصلی شما ساخت یک Collection، افزودن عناصر به آن و جستجو در آن است.
</div>

<a id='2'></a>
## ۲ - پیکربندی دیتابیس

---
<div dir="rtl">

در این بخش، شیء مرکزی این آزمایشگاه را بررسی می‌کنید: [Collection](https://docs.trychroma.com/docs/collections/manage-collections) - نامی که ChromaDB  به یک گروه از اشیای داده می‌دهد که برای بازیابی ایندکس می‌شوند.
گردش‌کار کلی همان چیزی است که در درس دیدید:
</div>
<div align="center">
  <img src="images/workflow.png" alt="RAG Overview" width="60%">
</div>


<a id='2-1'></a>
### ۲.۱ ساخت یک Collection

<div dir="rtl">
برای ساخت یک Collection، چند پارامتر مهم وجود دارد:

- `name`: نام Collection - همان نامی که در حافظه ذخیره می‌شود و برای بارگذاری دوباره‌ی آن استفاده خواهید کرد.
- `embedding_function`: تابعی که مسئول تبدیل متن به وکتور است. هر Collection در ChromaDB دقیقاً یک تابع embedding دارد که مسئول وکتورایز کردن تمام اسناد آن Collection است.
- `metadata`: یک دیکشنری اختیاری برای تنظیمات Collection، از جمله پیکربندی ایندکس HNSW (مثل فضای فاصله).

بیایید یک دیتابیس بارگذاری کنیم تا این بخش را نشان دهیم.
</div>

In [81]:
data = joblib.load("data.joblib")
print_object_properties(data[0])

place: دره ستارگان
state: هرمزگان
description: دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌های نمکی و فرسایشی که منظره‌ای شبیه سطح ماه دارد.
best_season_to_visit: پاییز, زمستان
attractions: صخره‌های ستاره‌ای, تورهای شب‌نوردی
budget: ارزان
user_ratings: 4.6
last_updated: 2025-01-01T00:00:00Z



In [83]:
print_object_properties(data[12])

place: باغ فین
state: اصفهان
description: باغ ایرانی تاریخی در کاشان با آب‌نماها و درختان کهن، ثبت‌شده در یونسکو.
best_season_to_visit: بهار
attractions: حوض‌های آب, عمارت تاریخی
budget: ارزان
user_ratings: 4.6
last_updated: 2025-01-01T00:00:00Z



In [85]:
len(data)

20

<div dir="rtl">

دیتاست شامل مجموعه‌ای از مکان‌های گردشگری است، همراه با ویژگی‌هایی که هر مکان را توصیف می‌کنند. ویژگی‌های موجود عبارت‌اند از: `place, state, description, best_season_to_visit, attractions, budget, user_ratings, last_updated`.


</div>

<a id='2-2'></a>
### ۲.۲ پیکربندی تابع Embedding

<div dir="rtl">
برای این‌که ChromaDB بداند چطور متن را به وکتور تبدیل کند، کافی است یک کلاس بسازید که رابط `EmbeddingFunction` را پیاده‌سازی کند - یعنی یک متد `__call__` داشته باشد که یک لیست از متن‌ها بگیرد و لیستی از وکتورها برگرداند. به‌صورت پیش‌فرض از `OllamaEmbeddings` در LangChain استفاده می‌کنیم که با مدل embedding محلی `bge-m3` روی سرویس Ollama کار می‌کند - مطمئن شوید Ollama روی سیستم شما در حال اجراست و این مدل را از قبل با `ollama pull bge-m3` دانلود کرده‌اید.

اگر ترجیح می‌دهید به‌جای مدل محلی از OpenAI استفاده کنید، کلاس جایگزین `OpenAIEmbeddingFunction` هم در سلول بعدی آماده است؛ کافی است `OPENAI_API_KEY` را در فایل `.env` تنظیم کنید و در انتهای سلول، `embedding_function` را به آن کلاس تغییر دهید.
</div>


In [93]:
class OllamaEmbeddingFunction(EmbeddingFunction):
    """تابع Embedding سازگار با ChromaDB که از OllamaEmbeddings در LangChain
    استفاده می‌کند - یک مدل embedding محلی که از طریق سرویس Ollama (روی
    http://localhost:11434 به‌صورت پیش‌فرض) فراخوانی می‌شود."""

    def __init__(self, model: str = "bge-m3"):
        self.embeddings = OllamaEmbeddings(model=model)

    def __call__(self, input: Documents) -> Embeddings:
        return self.embeddings.embed_documents(list(input))


class OpenAIEmbeddingFunction(EmbeddingFunction):
    """جایگزین: تابع Embedding سازگار با ChromaDB که از OpenAIEmbeddings در
    LangChain استفاده می‌کند. نیاز به OPENAI_API_KEY در فایل .env دارد."""

    def __init__(self, model: str = "text-embedding-3-small"):
        self.embeddings = OpenAIEmbeddings(model=model)

    def __call__(self, input: Documents) -> Embeddings:
        return self.embeddings.embed_documents(list(input))


# پیش‌فرض: Ollama با مدل محلی bge-m3
embedding_function = OllamaEmbeddingFunction()

# برای استفاده از OpenAI به‌جای آن، خط بالا را کامنت و خط زیر را فعال کنید:
# embedding_function = OpenAIEmbeddingFunction()


<div dir="rtl">
در ChromaDB هر شیء فقط **یک** رشته‌ی متنی (`document`) دارد که وکتورایز می‌شود. پس اگر بخواهیم چند فیلد را با هم وکتورایز کنیم، باید خودمان یک تابع کمکی بسازیم که فیلدهای مدنظر (`place, state, description, best_season_to_visit, attractions, budget`) را کنار هم می‌چسباند.
</div>

In [99]:
def build_document_text(record: Dict) -> str:
    """فیلدهایی که می‌خواهیم وکتورایز شوند را به هم می‌چسباند، چون Chroma فقط
    یک رشته را در document برای هر شیء وکتورایز می‌کند."""
    fields = ["place", "state", "description", "best_season_to_visit", "attractions", "budget"]
    return " ".join(f"{field}: {record[field]}" for field in fields)


print(build_document_text(data[0]))

place: دره ستارگان state: هرمزگان description: دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌های نمکی و فرسایشی که منظره‌ای شبیه سطح ماه دارد. best_season_to_visit: پاییز, زمستان attractions: صخره‌های ستاره‌ای, تورهای شب‌نوردی budget: ارزان


<a id='2-3'></a>
### ۲.۳ متادیتا (Metadata)

<div dir="rtl">
    
در یک Collection،
ویژگی‌های هر داده که وکتورایز **نمی‌شوند** ولی همراه هر شیء ذخیره می‌شوند، `metadata` نام دارند.
</div><div dir="rtl">
چند نکته‌ی مهم درباره‌ی متادیتا در ChromaDB:
    
- نیازی به تعریف اسکیمای از پیش‌تعیین‌شده برای هر فیلد نیست؛ هر دیکشنری پایتونی مستقیماً به‌عنوان متادیتا پذیرفته می‌شود.
- مقادیر متادیتا فقط می‌توانند از نوع `str`, `int`, `float` یا `bool` باشند (بدون `None` و بدون تودرتو بودن).
- ChromaDB نوع تاریخ ندارد؛ تاریخ‌ها را معمولاً به‌صورت رشته (ISO 8601) یا timestamp عددی ذخیره می‌کنید.
- متادیتا اصلاً وکتورایز نمی‌شود - فقط متن داخل `documents` وکتورایز می‌شود (همان چیزی که در `build_document_text` ساختیم).
</div>

In [105]:
COLLECTION_NAME = "trip"

# حذف Collection در صورت وجود، تا از صفر شروع کنیم
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

In [107]:
try:
    # اگر از قبل وجود داشته باشد، همان را بارگذاری می‌کنیم
    collection = client.get_collection(name=COLLECTION_NAME, embedding_function=embedding_function)
except Exception:
    # در غیر این صورت، یک Collection تازه می‌سازیم
    collection = client.create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_function,
        metadata={"hnsw:space": "cosine"},  # پیکربندی ایندکس HNSW (مثلاً فضای فاصله)
    )

اجرای این سلول یک Collection می‌سازد و آن را برمی‌گرداند. با پرینت کردن آن، پیکربندی Collection را می‌بینید.

In [109]:
print(collection)

Collection(name=trip)


<div dir="rtl">
اگر بخواهید Collection‌ای بسازید که از قبل وجود دارد، یک خطا صادر می‌شود:
</div>

In [111]:
try:
    collection = client.create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_function,
    )
except Exception as e:
    print(e)

Collection [trip] already exists


<div dir="rtl">
نکته: برای جلوگیری از این دست‌وپنجه نرم کردن با استثناها، ChromaDB متد راحت‌تری هم دارد: `client.get_or_create_collection(...)` که در یک فراخوانی، Collection را می‌سازد یا در صورت وجود بارگذاری می‌کند.

می‌توانید تمام Collection‌های ذخیره‌شده را هم بازیابی کنید:
</div>

In [113]:
client.list_collections()

[Collection(name=trip)]

<div dir="rtl">

نتیجه‌ی `list_collections()` لیستی از اشیای Collection است (هرکدام دارای `name`، `metadata` و ...).
</div>

<a id='2-4'></a>
### ۲.۴ افزودن عناصر به یک Collection

<div dir="rtl">
پس از ساخت یک Collection، آن خالی است. حالا باید عناصر را به آن اضافه کنید. هنگام افزودن هر عنصر، دو مرحله‌ی مهم در پس‌زمینه اتفاق می‌افتد:

۱. متن با فراخوانی `embedding_function` وکتورایز می‌شود

۲. ایندکس HNSW به‌روزرسانی می‌شود تا جستجو بهینه بماند (همان چیزی که در درس دیدید)

افزودن عناصر با متد `collection.add()` انجام می‌شود، که لیستی از `ids`، `documents` (متن‌هایی که وکتورایز می‌شوند) و `metadatas` (باقی فیلدها) می‌گیرد. با دادن یک شناسه‌ی یکتا (id) به هر عنصر، از ورودی‌های تکراری در دیتابیس جلوگیری می‌کنید.
</div>

In [121]:
def generate_id(record: Dict) -> str:
    """یک شناسه‌ی یکتا و قطعی از محتوای رکورد می‌سازد."""
    key = str(sorted(record.items()))
    return hashlib.md5(key.encode("utf-8")).hexdigest()

In [129]:
BATCH_SIZE = 8  # تعداد اشیایی که در هر فراخوانی add() ارسال می‌شوند

# فقط در صورتی داده اضافه می‌کنیم که Collection خالی باشد
if collection.count() == 0:
    for i in tqdm(range(0, len(data), BATCH_SIZE)):
        batch = data[i:i + BATCH_SIZE]
        collection.add(
            ids=[generate_id(record) for record in batch],
            documents=[build_document_text(record) for record in batch],
            metadatas=batch,  # چون هیچ فیلدی خودش وکتورایز نمی‌شود، کل رکورد را به‌عنوان متادیتا می‌فرستیم
        )

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:04<00:00,  1.37s/it]


عالی! حالا یک Collection با وکتور دارید! می‌توانید تعداد وکتورها را با `collection.count()` بررسی کنید:

In [131]:
collection.count()

20

<a id='3'></a>
## ۳ - جستجو روی یک کالکشن
<div dir="rtl">
در این بخش یاد می‌گیرید چطور روی یک Collection جستجو کنید. می‌توانید:

- روی متادیتا فیلتر کنید
- با جستجوی معنایی (Semantic Search) جستجو کنید
- با جستجوی کلیدواژه‌ای جستجو کنید
- جستجوی ترکیبی و بازرتبه‌بندی انجام دهید

بیایید چند مثال ببینیم.
</div>

<a id='3-1'></a>
### ۳.۱ فیلترها
<div dir="rtl">
پیش از پرداختن به جستجو، بیایید فیلترها را بررسی کنیم. فیلترها راهی برای محدود کردن جستجو بر اساس یک معیار هستند و بسیار انعطاف‌پذیرند. معمولاً آن‌ها را به‌عنوان یک آرگومان در یک جستجو پاس می‌دهید.

در ChromaDB فیلترها با پارامتر `where` و یک دیکشنری از عملگرها (`$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, `$and`, `$or`) مشخص می‌شوند.
</div>

In [133]:
# دو شیء را با فیلتر بر اساس ویژگی user_ratings واکشی می‌کنیم؛ فقط اشیایی که مقدارشان بزرگ‌تر یا مساوی 3.5 است
result = collection.get(limit=2, where={"user_ratings": {"$gte": 3.5}})

نتیجه یک دیکشنری پایتونی معمولی است:

In [135]:
result

{'ids': ['93b46399114660a053c0f7fbba71657b',
  '192285c66b01fd1fe999086ddf891c19'],
 'embeddings': None,
 'documents': ['place: دره ستارگان state: هرمزگان description: دره\u200cای شگفت\u200cانگیز در جزیره قشم با صخره\u200cهای نمکی و فرسایشی که منظره\u200cای شبیه سطح ماه دارد. best_season_to_visit: پاییز, زمستان attractions: صخره\u200cهای ستاره\u200cای, تورهای شب\u200cنوردی budget: ارزان',
  'place: چشمه\u200cهای آب\u200cگرم سرعین state: اردبیل description: منطقه\u200cای معروف به چشمه\u200cهای آب\u200cگرم طبیعی و درمانی در دامنه کوه سبلان. best_season_to_visit: تابستان, پاییز attractions: چشمه آب\u200cگرم گاومیش\u200cگلی, حمام\u200cهای سنتی budget: متوسط'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'place': 'دره ستارگان',
   'attractions': 'صخره\u200cهای ستاره\u200cای, تورهای شب\u200cنوردی',
   'last_updated': '2025-01-01T00:00:00Z',
   'budget': 'ارزان',
   'user_ratings': 4.6,
   'description': 'دره\u200cای شگفت\u200cانگیز در جزیره قشم با ص

می‌توانید متادیتای هر عنصر را از کلید `metadatas` بگیرید:

In [137]:
result["metadatas"]

[{'place': 'دره ستارگان',
  'attractions': 'صخره\u200cهای ستاره\u200cای, تورهای شب\u200cنوردی',
  'last_updated': '2025-01-01T00:00:00Z',
  'budget': 'ارزان',
  'user_ratings': 4.6,
  'description': 'دره\u200cای شگفت\u200cانگیز در جزیره قشم با صخره\u200cهای نمکی و فرسایشی که منظره\u200cای شبیه سطح ماه دارد.',
  'state': 'هرمزگان',
  'best_season_to_visit': 'پاییز, زمستان'},
 {'description': 'منطقه\u200cای معروف به چشمه\u200cهای آب\u200cگرم طبیعی و درمانی در دامنه کوه سبلان.',
  'last_updated': '2025-01-01T00:00:00Z',
  'best_season_to_visit': 'تابستان, پاییز',
  'user_ratings': 4.4,
  'budget': 'متوسط',
  'place': 'چشمه\u200cهای آب\u200cگرم سرعین',
  'state': 'اردبیل',
  'attractions': 'چشمه آب\u200cگرم گاومیش\u200cگلی, حمام\u200cهای سنتی'}]

پس، هر عنصر در این لیست، متادیتای یکی از اشیای بازگردانده‌شده است.

In [139]:
obj = result["metadatas"][0]

در ChromaDB، `obj` مستقیماً همان دیکشنری ویژگی‌هاست:

In [141]:
obj

{'place': 'دره ستارگان',
 'attractions': 'صخره\u200cهای ستاره\u200cای, تورهای شب\u200cنوردی',
 'last_updated': '2025-01-01T00:00:00Z',
 'budget': 'ارزان',
 'user_ratings': 4.6,
 'description': 'دره\u200cای شگفت\u200cانگیز در جزیره قشم با صخره\u200cهای نمکی و فرسایشی که منظره\u200cای شبیه سطح ماه دارد.',
 'state': 'هرمزگان',
 'best_season_to_visit': 'پاییز, زمستان'}

<div dir="rtl">

در این درس، روش فیلتر کردن `where={"property": {"$operator": value}}` است. در ادامه، نمونه‌های بیشتری از فیلتر کردن را همراه با روش‌های دیگر جستجو خواهید دید.
</div>

<a id='3-2'></a>
### ۳.۲ جستجوی معنایی (Semantic Search)
<div dir="rtl">
می‌توانید از جستجوی معنایی برای جستجو روی Collection استفاده کنید. این روش از وکتورها برای محاسبه‌ی فاصله بین آن‌ها استفاده می‌کند و نزدیک‌ترین‌ها را برمی‌گرداند. باید یک پرامپت پاس دهید که وکتورایز شده و با اشیای Collection مقایسه می‌شود. متد آن `.query()` است.
</div>

In [143]:
results = collection.query(
    query_texts=["می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. به دنبال مکان‌های ارزان هستم."],
    n_results=4,
)

In [145]:
# روی نتایج پیمایش می‌کنیم و متادیتای آن‌ها را چاپ می‌کنیم
# چون query_texts یک لیست می‌گیرد، نتایج هم یک لیست از لیست‌هاست - ایندکس [0] یعنی نتایج مربوط به اولین پرامپت
for obj in results["metadatas"][0]:
    print_object_properties(obj)

attractions: کاروانسرای مرنجاب, دریاچه نمک
last_updated: 2025-01-01T00:00:00Z
best_season_to_visit: پاییز, زمستان
state: اصفهان
user_ratings: 4.4
description: کویری خاموش و دیدنی با تپه‌های شنی و دریاچه نمک، مناسب برای سفرهای صحرایی.
place: کویر مرنجاب
budget: متوسط

state: هرمزگان
attractions: کشتی یونانی, دلفیناریوم
description: جزیره‌ای مرجانی در خلیج فارس، معروف به سواحل زیبا و مراکز خرید.
budget: گران
user_ratings: 4.5
best_season_to_visit: زمستان
place: جزیره کیش
last_updated: 2025-01-01T00:00:00Z

attractions: آبشارهای فصلی, پل‌های معلق
user_ratings: 4.7
last_updated: 2025-01-01T00:00:00Z
description: دره‌ای صخره‌ای و سرسبز با آبشارهای فصلی و مسیرهای کوهنوردی.
state: لرستان
budget: ارزان
place: تنگه واشی
best_season_to_visit: بهار

place: دره ستارگان
attractions: صخره‌های ستاره‌ای, تورهای شب‌نوردی
user_ratings: 4.6
state: هرمزگان
budget: ارزان
best_season_to_visit: پاییز, زمستان
description: دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌های نمکی و فرسایشی که منظره‌ای شبیه سطح ماه دارد.

<div dir="rtl">

می‌توانید هم‌زمان روی عناصری که `budget = ارزان` دارند هم فیلتر کنید:
</div>

In [147]:
results = collection.query(
    query_texts=["می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. "],
    where={"budget": "ارزان"},   # فیلتر برابری روی متادیتا
    n_results=4,
)

In [149]:
for obj in results["metadatas"][0]:
    print_object_properties(obj)

state: لرستان
place: تنگه واشی
attractions: آبشارهای فصلی, پل‌های معلق
best_season_to_visit: بهار
budget: ارزان
user_ratings: 4.7
last_updated: 2025-01-01T00:00:00Z
description: دره‌ای صخره‌ای و سرسبز با آبشارهای فصلی و مسیرهای کوهنوردی.

description: دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌های نمکی و فرسایشی که منظره‌ای شبیه سطح ماه دارد.
budget: ارزان
last_updated: 2025-01-01T00:00:00Z
state: هرمزگان
attractions: صخره‌های ستاره‌ای, تورهای شب‌نوردی
place: دره ستارگان
user_ratings: 4.6
best_season_to_visit: پاییز, زمستان

last_updated: 2025-01-01T00:00:00Z
attractions: قلعه دژ, حمام و مسجد تاریخی
budget: ارزان
place: ارگ بم
description: بزرگ‌ترین سازه خشتی جهان، دژی تاریخی با قدمتی بیش از دو هزار سال.
user_ratings: 4.5
best_season_to_visit: پاییز, زمستان
state: کرمان

description: پلی تاریخی و چندمنظوره روی زاینده‌رود با معماری دوران صفوی.
attractions: طاق‌های تزئینی, نشستن کنار رودخانه
state: اصفهان
user_ratings: 4.8
best_season_to_visit: بهار, پاییز
place: پل خواجو
budget: ارزان
last_

<div dir="rtl">
همچنین می‌توانید لیستی از مقادیر ممکن را در یک فیلتر پاس دهید، با استفاده از عملگر `$in`:
</div>

In [151]:
results = collection.query(
    query_texts=["می‌خواهم برای سفر زمستانی پیشنهاد بگیرم."],
    where={"budget": {"$in": ["ارزان", "متوسط"]}},
    n_results=4,
)

In [153]:
for obj in results["metadatas"][0]:
    print_object_properties(obj)

budget: متوسط
last_updated: 2025-01-01T00:00:00Z
state: اصفهان
user_ratings: 4.4
place: کویر مرنجاب
best_season_to_visit: پاییز, زمستان
description: کویری خاموش و دیدنی با تپه‌های شنی و دریاچه نمک، مناسب برای سفرهای صحرایی.
attractions: کاروانسرای مرنجاب, دریاچه نمک

last_updated: 2025-01-01T00:00:00Z
best_season_to_visit: بهار
user_ratings: 4.7
description: دره‌ای صخره‌ای و سرسبز با آبشارهای فصلی و مسیرهای کوهنوردی.
attractions: آبشارهای فصلی, پل‌های معلق
state: لرستان
budget: ارزان
place: تنگه واشی

attractions: ساحل رنگین‌کمان, غار نمکی
budget: متوسط
state: هرمزگان
best_season_to_visit: زمستان
description: جزیره‌ای رنگارنگ با خاک‌های اُخرایی، معروف به ساحل رنگین‌کمان.
place: جزیره هرمز
last_updated: 2025-01-01T00:00:00Z
user_ratings: 4.6

place: دره ستارگان
budget: ارزان
last_updated: 2025-01-01T00:00:00Z
state: هرمزگان
best_season_to_visit: پاییز, زمستان
user_ratings: 4.6
attractions: صخره‌های ستاره‌ای, تورهای شب‌نوردی
description: دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌های نمکی و 

<a id='3-3'></a>
### ۳.۳ جستجوی کلیدواژه‌ای (Keyword / Full-Text Search)

<div dir="rtl">
برای جستجوی کلیدواژه‌ای روی متن اسناد، از پارامتر `where_document` با عملگر `$contains` استفاده می‌کنید.

<div class="alert alert-block alert-warning">
<b>نکته‌ی مهم:</b> نسخه‌ی محلی/متن‌باز ChromaDB به‌صورت پیش‌فرض هیچ رتبه‌بندی BM25 ندارد؛ <code>where_document</code> فقط یک <b>فیلتر true/false</b> است (آیا رشته در متن وجود دارد یا نه)، نه یک الگوریتم امتیازدهی. رتبه‌بندی واقعی BM25/sparse فقط در Chroma Cloud با پیکربندی یک ایندکس sparse vector در دسترس است. در بخش بعد (۳.۴) نشان می‌دهیم چطور می‌توانید خودتان یک رتبه‌بندی BM25 محلی بسازید.
</div>
</div>

In [ ]:
result = collection.get(
    where_document={"$contains": "تاریخی"},
    where={"budget": {"$in": ["ارزان", "متوسط"]}},  # می‌توان where و where_document را با هم ترکیب کرد
    limit=4,
)

In [ ]:
for obj in result["metadatas"]:
    print_object_properties(obj)

<a id='3-4'></a>
### ۳.۴ جستجوی ترکیبی (Hybrid Search)

<div dir="rtl">
این همان جستجوی RRF (Reciprocal Rank Fusion) است که در درس دیدید: ترکیب رتبه‌بندی جستجوی برداری (معنایی) و رتبه‌بندی کلیدواژه‌ای.

چون نسخه‌ی محلی ChromaDB رتبه‌بندی BM25 داخلی ندارد (بخش ۳.۳)، اینجا خودمان یک رتبه‌بندی کلیدواژه‌ای محلی با کتابخانه‌ی سبک `rank_bm25` می‌سازیم و آن را با رتبه‌بندی برداری Chroma، با همان فرمول Reciprocal Rank Fusion، ترکیب می‌کنیم.
</div>

In [ ]:
from rank_bm25 import BM25Okapi

# یک ایندکس BM25 محلی روی تمام اسناد Collection می‌سازیم (رتبه‌بندی کلیدواژه‌ای سمت کلاینت،
# چون رتبه‌بندی BM25/sparse بومی ChromaDB یک قابلیت Chroma Cloud است)
all_data = collection.get(include=["documents", "metadatas"])
corpus = all_data["documents"]
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


def bm25_rank(query: str, top_k: int = 10) -> List[str]:
    """رتبه‌بندی کلیدواژه‌ای محلی؛ فهرستی از idها را از بهترین به بدترین برمی‌گرداند."""
    scores = bm25.get_scores(query.lower().split())
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [all_data["ids"][i] for i in ranked_idx]

In [ ]:
def reciprocal_rank_fusion(rankings: List[List[str]], weights: List[float], k: int = 60) -> List[str]:
    """چند فهرست رتبه‌بندی‌شده از idها را با فرمول وزن‌دار Reciprocal Rank Fusion
    ترکیب می‌کند - همان الگوریتمی که Rrf() در Chroma Cloud روی dense + sparse
    اعمال می‌کند."""
    scores: Dict[str, float] = {}
    for ranking, weight in zip(rankings, weights):
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + weight / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

In [ ]:
query = "می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. به دنبال مکان‌های ارزان هستم."

dense_result = collection.query(query_texts=[query], n_results=10)
dense_rank = dense_result["ids"][0]           # رتبه‌بندی برداری (معنایی)
keyword_rank = bm25_rank(query, top_k=10)     # رتبه‌بندی کلیدواژه‌ای محلی

# ترکیب: مثلاً ۷۰٪ وزن جستجوی برداری، ۳۰٪ وزن جستجوی کلیدواژه‌ای
hybrid_ids = reciprocal_rank_fusion([dense_rank, keyword_rank], weights=[0.7, 0.3])

hybrid_result = collection.get(ids=hybrid_ids[:4], include=["metadatas"])
for obj in hybrid_result["metadatas"]:
    print_object_properties(obj)

<div dir="rtl">
<div class="alert alert-block alert-info">
<b>نکته:</b> اگر از Chroma Cloud استفاده می‌کردید و روی Collection یک ایندکس sparse vector پیکربندی می‌کردید، می‌توانستید همین کار را با API داخلی جدید Chroma انجام دهید:

<div dir="ltr">

```python
from chromadb import Search, K, Knn, Rrf

hybrid_rank = Rrf(
    ranks=[
        Knn(query=query, limit=10),                          # رتبه‌بندی برداری
        Knn(query=query, key="sparse_embedding", limit=10),  # رتبه‌بندی sparse/BM25
    ],
    weights=[0.7, 0.3],
    k=60,
)
search = Search().rank(hybrid_rank).limit(4).select(K.DOCUMENT, K.SCORE)
results = collection.search(search)
```
</div>

اما چون این قابلیت به Chroma Cloud نیاز دارد، در این آزمایشگاه از پیاده‌سازی محلی بالا استفاده کردیم که همان مفهوم را بدون نیاز به سرویس ابری نشان می‌دهد.
</div>
</div>

<a id='3-5'></a>
### ۳.۵ بازرتبه‌بندی (Reranking)
<div dir="rtl">
ChromaDB ماژول بازرتبه‌بندی داخلی ندارد. اما چون بازرتبه‌بندی صرفاً یک مرحله‌ی پس‌پردازش روی نتایج بازیابی‌شده است، به‌سادگی می‌توانیم یک مدل cross-encoder را مستقیماً در پایتون فراخوانی کنیم - بدون نیاز به هیچ سروری.
    </div>

In [ ]:
from FlagEmbedding import FlagReranker

reranker = FlagReranker(
    "BAAI/bge-reranker-base",
    use_fp16=False,
)


def rerank(query: str, candidates: List[Dict], text_key: str = "attractions", top_k: int = None):
    """بازرتبه‌بندی cross-encoder روی یک لیست از رکوردهای متادیتا."""
    pairs = [(query, candidate[text_key]) for candidate in candidates]
    scores = reranker.compute_score(pairs)
    if not isinstance(scores, list):
        scores = [scores]
    return sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)[:top_k]

In [ ]:
# اول با جستجوی معنایی چند کاندید می‌گیریم
response = collection.query(
    query_texts=["می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. به دنبال مکان‌های ارزان و سرگرم‌کننده هستم."],
    n_results=5,
)
candidates = response["metadatas"][0]

# سپس این کاندیدها را بر اساس فیلد attractions و پرامپت جدید 'مکان‌های سرگرم‌کننده' بازرتبه‌بندی می‌کنیم
reranked = rerank(query="مکان‌های سرگرم‌کننده", candidates=candidates, text_key="attractions")

for obj, score in reranked:
    print(f"score={score:.4f}")
    print_object_properties(obj)

نکته: `PersistentClient` کروما هیچ زیرفرایند جداگانه‌ای اجرا نمی‌کند؛ بستن صریح لازم نیست، اما داده‌های شما در مسیری که مشخص کردید (`./.collections`) ماندگار باقی می‌مانند.